# Matrix inversion with fresh QSVT phases

Regenerate the original model and compare the explicit and precision-driven inverses.

Use the local FlagQuantum environment.

In [ ]:
from pathlib import Path

import torch
from IPython.display import Image, display

import flagquantum as fq

Regenerate the original seeded matrix and normalize its spectral norm.

In [ ]:
from examples.synthesis.algorithms.quantum_linear_solvers.qsvt_matrix_inversion.overview import (
    model,
    workflow,
)
from examples.synthesis.algorithms.quantum_linear_solvers.qsvt_matrix_inversion.overview.main import (
    print_report,
    save_response,
    save_solution,
)

print(fq.__name__)
fresh = model.build_model()
print(fresh.matrix.real)

Inspect the singular values.

In [ ]:
print(fresh.spectrum)

Use all sixteen Pauli terms and four block bits.

In [ ]:
print("Data bits, selector bits and block scale:", 2, 4, fresh.encoding.alpha)

Keep the original bare and block-encoded condition numbers.

In [ ]:
print("Bare and effective condition numbers:", fresh.condition, fresh.effective_condition)

Calculate the odd degree and the target inverse coefficients.

In [ ]:
degree = workflow.manual_degree(fresh)
coefficients = workflow.inverse_coefficients(fresh, degree)
print("Explicit polynomial degree:", degree)

Prepare the original right-hand side.

In [ ]:
rhs = workflow.source_rhs()
print(rhs)

Apply the newly fitted symmetric phase sequence.

In [ ]:
manual = workflow.solve(fresh, rhs, degree)
print("Explicit phase count:", manual.phase_fit.phases.numel())

Measure the polynomial response and the full complex-state error.

In [ ]:
print(manual.response_error, manual.full_state_error)

Inspect the observed zero-block probability.

In [ ]:
print("Explicit projection probability:", manual.projection_probability)

Compute the comparison vector by a direct solve.

In [ ]:
expected = torch.linalg.solve(fresh.matrix, rhs.to(fresh.matrix.dtype)).real

Recover the signed solution from the zero-block state.

In [ ]:
print(manual.solution)

Plot the inverse response and the approximation residual.

In [ ]:
figures = Path("notebook-figures")
save_response(fresh, manual, figures / "inverse-response.png")
display(Image(filename=str(figures / "inverse-response.png")))

Check the original 0.05 solution tolerance.

In [ ]:
assert float((expected - manual.solution).norm()) < 0.05

Run the precision-driven inverse and compare all solution vectors.

In [ ]:
direct_degree = workflow.inverse_degree(fresh, workflow.AUTOMATIC_ERROR)
automatic = workflow.solve(fresh, rhs, direct_degree)
print_report(fresh, manual, automatic)
save_solution(fresh, manual, automatic, figures / "matrix-solution.png")
display(Image(filename=str(figures / "matrix-solution.png")))